## Setup

In [ ]:
import pandas as pd  # tables of data

data_path = "/content/drive/My Drive/neet_data.csv"  # where the cleaned data lives on Drive

## Mount Google Drive

In [ ]:
from google.colab import drive  # Colab's Google Drive tool
drive.mount("/content/drive")  # connect Google Drive

## Load the data

In [ ]:
df = pd.read_csv(data_path)  # load the cleaned data into a table

## 2026: Education of NEET people by age group and sex

In [ ]:
if "agegrp" not in df.columns:  # make age groups only if the file does not have them
    df["agegrp"] = pd.cut(df["age"], bins=[16, 20, 25, 30], labels=["16-19", "20-24", "25-29"], right=False)  # 16-19, 20-24, 25-29

df["sex"] = pd.Categorical(df["female"].map({0: "Men", 1: "Women"}), categories=["Men", "Women"], ordered=True)  # words for female, Men first

educ_labels = ["Less than high school", "High school or GED", "Some college", "Bachelor or higher"]  # education categories, in order
if "educ4" not in df.columns:  # rebuild educ4 from the four 0/1 education variables if the file does not have it
    educ4 = pd.Series(pd.NA, index=df.index, dtype="object")  # start with everyone missing
    for column, label in zip(["less_hs", "hs_ged", "some_college", "bachelor_plus"], educ_labels):  # one 0/1 variable per category
        educ4[df[column] == 1] = label  # people with a 1 get this category
    df["educ4"] = pd.Categorical(educ4, categories=educ_labels, ordered=True)  # ordered categories

neet26 = df[(df["year"] == 2026) & (df["neet"] == 1)].copy()  # keep only 2026 and only NEET people
n_left_out = neet26["educ4"].isna().sum()  # NEET people with a missing educ4
known = neet26.dropna(subset=["educ4"])  # leave them out

row_order = [f"{age} {sex}" for age in ["16-19", "20-24", "25-29"] for sex in ["Men", "Women"]]  # the six rows, in order
keys = ["agegrp", "sex", "educ4"]  # group by age group, sex, education

weights = known.groupby(keys, observed=True)["asecwt"].sum().unstack("educ4")  # sum of asecwt in each cell
weights.index = [f"{age} {sex}" for age, sex in weights.index]  # row labels like "16-19 Men"
weights = weights.reindex(index=row_order, columns=educ_labels).fillna(0)  # rows and columns in our order, empty cells = 0
table = weights.div(weights.sum(axis=1), axis=0) * 100  # percent of NEET people in each row

people = known.groupby(keys, observed=True).size().unstack("educ4")  # unweighted respondents in each cell
people.index = [f"{age} {sex}" for age, sex in people.index]  # same row labels as the table
people = people.reindex(index=row_order, columns=educ_labels).fillna(0).astype(int)  # same layout, empty cells = 0

shown = table.copy().astype(object)  # text version of the table, for the asterisks
for row in table.index:  # loop over the rows
    for column in table.columns:  # loop over the columns
        star = "*" if people.loc[row, column] < 30 else " "  # asterisk if fewer than 30 respondents
        shown.loc[row, column] = f"{table.loc[row, column]:.1f}{star}"  # percent with 1 decimal, then the asterisk or a space

row_sums = table.sum(axis=1)  # each row's total, before rounding
off = (row_sums - 100).abs()  # distance from 100 in each row

print("NEET people in 2026 left out because educ4 is missing (unweighted):", n_left_out)  # how many were left out
print()  # empty line
print("Table. Education of NEET people in 2026, % of NEET people in each row (weighted with asecwt).")  # table title
print(shown.to_string())  # table, 1 decimal, asterisks
print("* fewer than 30 unweighted respondents")  # key for the asterisk
print()  # empty line
print("Check: every row adds up to 100.0:", bool((off < 1e-9).all()))  # True or False
print("Largest distance from 100 in any row (before rounding):", off.max())  # should be about 0
print()  # empty line
print("Unweighted number of respondents in each cell.")  # counts title
print(people.to_string())  # counts, same layout

**What one cell means.** Take the row "20-24 Women" and the column "Some college". Among women aged 20-24 in the 2026 survey who were NEET (not employed and not in school) and whose education is known, the cell is the percent who had some college. Each row adds up to 100, so the cells show how NEET people in that row are split across the four education levels. This is a share of NEET people, not a share of all women aged 20-24, so it is not a NEET rate. People are counted by how many people in the country they represent (`asecwt`). An asterisk marks a cell with fewer than 30 survey respondents behind it, shown in the count table under the table.